# Chapter 14 &mdash; Combining Semi-Deciders, and the "No Wimp" Clause

**Concept 7 of the Chapter 14 decomposition:** *Combining Semi-Deciders for $L$ and $\overline{L}$, and the "No Wimp" Clause*

If $L$ and $\overline{L}$ are both RE then $L$ is recursive &mdash; run both and take whichever answers.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Combining-Semi-Deciders/Concept-Combining-Semi-Deciders.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


> **Theorem.** $L$ is recursive $\iff$ $L$ and $\overline{L}$ are both RE.

($\Rightarrow$) A decider is trivially a semi-decider for both.

($\Leftarrow$) **Dovetail.** Run the two semi-deciders in lock-step, one step each in
turn. Exactly one of them must eventually halt &mdash; every string is in $L$ or in
$\overline{L}$ &mdash; so the combination always answers.

Two details matter. You must **interleave**: running one to completion first can hang
forever. And the book's **"no wimp" clause**: the combined machine may not shrug. It
must commit to an answer, which is precisely what dovetailing guarantees.

The contrapositive is the workhorse: if $L$ is RE but **not** recursive, then
$\overline{L}$ is **not** RE.

## 2. Definitions

### Two semi-deciders for complementary languages

In [ ]:
Yes = md2mc('''TM
!! semi-decides "starts with 1"
I : 1 ; 1 , R -> F
I : 0 ; 0 , R -> L
I : . ; . , R -> L
L : 0 ; 0 , R -> L
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')
No = md2mc('''TM
!! semi-decides "does NOT start with 1"
I : 0 ; 0 , R -> F
I : . ; . , R -> F
I : 1 ; 1 , R -> L
L : 0 ; 0 , R -> L
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')

# --- thin wrappers over Jove's TM runner --------------------------------
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

### Dovetailing, and the wrong way to do it

In [ ]:
def dovetail(A, B, tape, step=3, cap=300):
    for fuel in range(step, cap + 1, step):
        if tm_halts(A, tape, fuel=fuel): return ('in L', fuel)
        if tm_halts(B, tape, fuel=fuel): return ('not in L', fuel)
    return ('no answer', cap)

def sequential(A, B, tape, cap=300):
    # WRONG: run A to completion first
    if tm_halts(A, tape, fuel=cap): return ('in L', cap)
    return ('gave up on A; never started B', cap)

## 3. Tests

Dovetailing always answers.

In [ ]:
for t in ['1', '0', '11', '00', '10', '01']:
    verdict, fuel = dovetail(Yes, No, t)
    print("  %-5r -> %-10s after fuel %d" % (t, verdict, fuel))
    assert verdict != 'no answer'

And it answers **correctly**.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(1, 5) for p in product('01', repeat=k)]
for t in strs:
    verdict, _ = dovetail(Yes, No, t)
    assert (verdict == 'in L') == t.startswith('1'), t
print("dovetailing decides correctly on all %d strings" % len(strs))

**Interleaving is essential.** Running one first can hang forever.

In [ ]:
print("  sequential on '0' :", sequential(Yes, No, '0'))
print("  dovetail   on '0' :", dovetail(Yes, No, '0'))
print()
print("A never halts on '0', so the sequential version never reaches B.")
assert sequential(Yes, No, '0')[0].startswith('gave up')
assert dovetail(Yes, No, '0')[0] == 'not in L'

The **'no wimp' clause**: the combined machine must commit.

In [ ]:
print("It is not allowed to output 'I do not know'.")
print("Dovetailing guarantees it never has to: every string is in L or in")
print("its complement, so one of the two semi-deciders WILL halt.")

The contrapositive, which is how the theorem is actually used.

In [ ]:
print("L recursive  <=>  L RE  and  complement(L) RE")
print()
print("so:  L is RE but NOT recursive  =>  complement(L) is NOT RE")
print()
print("A_TM is RE and not recursive (Concept 10, Chapter 15),")
print("therefore complement(A_TM) is not even RE.  That is how you exhibit")
print("a language outside RE without any counting argument.")

## 4. Exercises


1. Why must the interleaving give each machine a *bounded* slice at a time?
2. If $L$ and $\overline L$ are both RE, how big is the decider you get?
3. Name a language where you know $L$ is RE and suspect $\overline L$ is not.

In [ ]:
# Your work for the exercises above.